# Lab 07 — 00 Source Preparation

Downloads Chicago data, injects controlled quality fixtures, and creates canonical snapshot versions.

**Storage design:** Lab 07 uses a Unity Catalog **EXTERNAL Volume** backed by ADLS Gen2. The notebook validates that the Volume is external before writing source files.

In [0]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / 'src' / 'lab07').exists()),
    None,
)

if project_root and str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))
if project_root and str(project_root / 'tools') not in sys.path:
    sys.path.insert(0, str(project_root / 'tools'))

dbutils.widgets.text('catalog', 'dbr_dev', '01 Catalog')
dbutils.widgets.text('schema', 'parvinbadalov', '02 Schema')
dbutils.widgets.text('volume_name', 'lab07_data_quality', '03 External Volume')
dbutils.widgets.text('storage_account', 'dlspl21databricks', '04 Storage Account')
dbutils.widgets.text('container', 'parvinbadalov', '05 ADLS Container')
dbutils.widgets.text('external_volume_dir', 'lab07_data_quality', '06 External Volume Directory')
dbutils.widgets.text('run_id', 'manual', '07 Run ID')

catalog = dbutils.widgets.get('catalog').strip()
schema = dbutils.widgets.get('schema').strip()
volume_name = dbutils.widgets.get('volume_name').strip()
storage_account = dbutils.widgets.get('storage_account').strip()
container = dbutils.widgets.get('container').strip()
external_volume_dir = dbutils.widgets.get('external_volume_dir').strip().strip('/')
run_id = dbutils.widgets.get('run_id').strip()

assert catalog == 'dbr_dev' and schema == 'parvinbadalov', (
    f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
)

volume_fqn = f'{catalog}.{schema}.{volume_name}'
volume_root = f'/Volumes/{catalog}/{schema}/{volume_name}'
external_volume_url = (
    f'abfss://{container}@{storage_account}.dfs.core.windows.net/'
    f'{external_volume_dir}'
)

print(f'External Volume FQN : {volume_fqn}')
print(f'External ADLS path  : {external_volume_url}')


## Create and validate the external Volume

The external location/storage credential must already permit access to the ADLS path. If a managed Volume with the same name already exists, this notebook deliberately fails rather than silently using it.

In [0]:
from license_batch_loader import download
from pyspark.sql import functions as F

dbutils.widgets.text('source_from_date', '2024-01-01', '08 Source From')
dbutils.widgets.text('source_max_rows', '300000', '09 Max Rows')
dbutils.widgets.text(
    'snapshot_cutoffs',
    '2024-12-31,2025-12-31,2026-08-15',
    '10 Snapshot Cutoffs',
)

source_from_date = dbutils.widgets.get('source_from_date').strip()
source_max_rows = int(dbutils.widgets.get('source_max_rows'))
cutoffs = [
    x.strip()
    for x in dbutils.widgets.get('snapshot_cutoffs').split(',')
    if x.strip()
]

spark.sql(
    f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`
    LOCATION '{external_volume_url}'
    """
)

volume_df = spark.sql(f'DESCRIBE VOLUME {volume_fqn}')
display(volume_df)

volume_info = volume_df.first().asDict()
volume_type = str(volume_info.get('volume_type', '')).upper()
storage_location = volume_info.get('storage_location')

if volume_type != 'EXTERNAL':
    raise ValueError(
        f'{volume_fqn} must be EXTERNAL, but found {volume_type or "UNKNOWN"}. '
        'Do not continue with the managed Volume.'
    )

if not storage_location:
    raise ValueError(f'{volume_fqn} does not expose a storage_location.')

print('✅ External volume validation passed.')
print(f'Storage location: {storage_location}')

landing = f'{volume_root}/landing/events'
dbutils.fs.rm(landing, True)
dbutils.fs.mkdirs(landing)

manifest = download(
    landing,
    source_from_date=source_from_date,
    max_rows=source_max_rows,
)
print(manifest)


In [0]:
from lab07.transformations import prepare_business_licenses
from lab07.quality_rules import classify_license_records
from lab07.snapshot_policy import canonical_snapshot, assert_unique_snapshot

raw = spark.read.json(f'{volume_root}/landing/events/*.json')
assert raw.count() > 0

# Add deterministic DQ fixtures that are easy to distinguish from real source rows.
sample = raw.filter('id IS NOT NULL AND license_number IS NOT NULL').limit(4).collect()
fixtures = []

if len(sample) >= 4:
    def one(r):
        return spark.createDataFrame([r], schema=raw.schema)

    fixtures = [
        one(sample[0])
        .withColumn('id', F.lit('LAB07_TEST_BAD_STATUS'))
        .withColumn('license_status', F.lit('BAD'))
        .withColumn('_fixture_kind', F.lit('INVALID_LICENSE_STATUS')),
        one(sample[1])
        .withColumn('id', F.lit('LAB07_TEST_BAD_ZIP'))
        .withColumn('zip_code', F.lit('ABC'))
        .withColumn('_fixture_kind', F.lit('INVALID_ZIP')),
        one(sample[2])
        .withColumn('id', F.lit('LAB07_TEST_WARN_DBA'))
        .withColumn('doing_business_as_name', F.lit(None).cast('string'))
        .withColumn('_fixture_kind', F.lit('WARN_DBA_MISSING')),
        one(sample[3])
        .withColumn('id', F.lit('LAB07_TEST_BAD_DATES'))
        .withColumn('license_start_date', F.lit('2026-12-31T00:00:00'))
        .withColumn('expiration_date', F.lit('2026-01-01T00:00:00'))
        .withColumn('_fixture_kind', F.lit('EXPIRATION_BEFORE_START')),
    ]

landed = raw
for fixture_df in fixtures:
    landed = landed.unionByName(fixture_df, allowMissingColumns=True)

landed.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(
    f'{catalog}.{schema}.business_license_landing'
)

classified = classify_license_records(prepare_business_licenses(landed))
source = classified.filter(
    "_dq_status <> 'QUARANTINE' AND _fixture_kind IS NULL"
)

frames = []
for version, cutoff in enumerate(cutoffs, 1):
    snap = canonical_snapshot(source, cutoff)
    assert_unique_snapshot(snap)
    frames.append(
        snap
        .withColumn('snapshot_version', F.lit(version))
        .withColumn('snapshot_cutoff', F.to_timestamp(F.lit(cutoff)))
    )

feed = frames[0]
for frame in frames[1:]:
    feed = feed.unionByName(frame, allowMissingColumns=True)

feed.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(
    f'{catalog}.{schema}.business_license_snapshot_feed'
)

display(
    feed.groupBy('snapshot_version').agg(
        F.count('*').alias('rows'),
        F.countDistinct('license_number').alias('distinct_keys'),
    )
)

print('✅ SOURCE PREPARATION COMPLETE')


In [0]:
%sql
SELECT COUNT(*)
FROM dbr_dev.parvinbadalov.business_license_landing;

In [0]:
%sql
SELECT
    snapshot_version,
    COUNT(*) AS rows,
    COUNT(DISTINCT license_number) AS distinct_keys
FROM dbr_dev.parvinbadalov.business_license_snapshot_feed
GROUP BY snapshot_version
ORDER BY snapshot_version;